# Yeast promoter design with TACO on Google Colab, using PyTorch

This notebook prepares the published TACO code<sup>1</sup> so that it can run on Google Colab. **Part 1** holds the setup steps only: imports, settings, the graphics card, Google Drive, unpacking the code, checking the files, and finding the file paths that must be changed.

Not written yet: downloading the model and data files from Hugging Face, changing the file paths, the tiny test run with its timing, and the extra checker networks. These parts are added after the Hugging Face file lists are known.

### **Step 1: Package Import**

Every import is in this one cell. The cell then prints the version of the packages that are already installed. On a fresh Colab session, some of them may be missing, which is expected at this point.

Expected printout: one line per package, with its version or the words *not installed yet*.

In [ ]:
from importlib.metadata import version, PackageNotFoundError
import pathlib
import platform
import shutil
import subprocess
import zipfile

print("Python==" + platform.python_version())

packages = ["torch", "numpy", "pandas"]

for package in packages:
    try:
        print(package + "==" + version(package))
    except PackageNotFoundError:
        print(package + ": not installed yet")

### **Step 2: Settings**

All names and folders are written down in this one cell. If a name changes, it is changed here only.

In [ ]:
# SETTINGS: the only cell that should need editing.

# The folder in your Google Drive that holds everything big.
drive_project_folder_name = "Reinforcement_Learning_for_Promoter_Design"

# The subfolder of that Drive folder, and the name of the zip file inside it.
code_subfolder_name = "taco_code"
taco_zip_file_name = "TACO-main.zip"

# The working folder for this session. Colab erases this folder when the session ends.
working_folder_name = "/content/working"

# Leave this empty on Colab. Only for testing outside Colab: a folder that stands in for the Drive folder.
local_test_drive_folder = ""

### **Step 3: Check the graphics card**

TACO is written for a graphics card from NVIDIA, so it cannot run without one. The command `nvidia-smi` asks the computer which card it has.

Expected printout: on Colab with a graphics card, a table that names the card and its memory (your earlier session showed a Tesla T4 with about 15 gigabytes). Without a card, the cell says so.

In [ ]:
# HELPER FUNCTION
# Asking the computer for its graphics card and printing the answer.
def show_graphics_card_information():
    try:
        completed_process = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        print(completed_process.stdout)
    except FileNotFoundError:
        print("No graphics card found. On Colab: Runtime, Change runtime type, choose a graphics card.")

show_graphics_card_information()

### **Step 4: Connect Google Drive**

Colab erases its own files when the session ends, but Drive keeps them. So the zip, the downloaded model files and the results all live in the Drive folder.

Expected printout: on Colab, a permission window opens first; click through it. Then two lines: the Drive folder path, and `Folder exists: True`.

In [ ]:
# HELPER FUNCTION
# Connecting Drive on Colab, and returning the project folder in it.
def find_drive_project_folder():
    if local_test_drive_folder != "":
        # Only used for testing outside Colab.
        return pathlib.Path(local_test_drive_folder)
    from google.colab import drive
    drive.mount("/content/drive")
    return pathlib.Path("/content/drive/MyDrive") / drive_project_folder_name

drive_project_folder = find_drive_project_folder()
print("Drive project folder:", drive_project_folder)
print("Folder exists:", drive_project_folder.exists())

### **Step 5: Unpack the TACO zip into the working folder**

The zip is copied from Drive and unpacked in the working folder. Working from the session's own disk is faster than working from Drive, and the zip in Drive stays untouched. If the working folder already exists, it is deleted first, so that old changes are never mixed in.

Expected printout: the number of files unpacked (117 for the zip used here), and the unpacked folder `TACO-main` with `exists: True`.

In [ ]:
taco_zip_path = drive_project_folder / code_subfolder_name / taco_zip_file_name
working_folder = pathlib.Path(working_folder_name)

# Stopping early, with a clear message, if the zip is not where Step 2 says.
if not taco_zip_path.exists():
    raise FileNotFoundError("The zip was not found at " + str(taco_zip_path) + ". Check the names in Step 2.")

if working_folder.exists():
    shutil.rmtree(working_folder)
working_folder.mkdir(parents=True)

with zipfile.ZipFile(taco_zip_path) as zip_file:
    zip_file.extractall(working_folder)
    print("Files unpacked:", len(zip_file.namelist()))

taco_folder = working_folder / "TACO-main"
print("Unpacked folder:", taco_folder, "| exists:", taco_folder.exists())

### **Step 6: Check that the expected files are there**

This looks for the files that the TACO instructions and code mention, to find out early whether the zip is complete.

Expected printout: one line per file, each ending in `True`.

In [ ]:
expected_files = [
    "README.md",
    "reinforce_mbo.py",
    "reinforce_mbo.sh",
    "env_install.sh",
    "dna_optimizers/mbo_optimizaer.py",
    "dna_optimizers/experience.py",
    "reglm_src/reglm/regression.py",
    "reglm_src/reglm/lightning.py",
]
for relative_file_name in expected_files:
    print(relative_file_name, (taco_folder / relative_file_name).exists())

### **Step 7: Find the file paths that belong to the authors' computer**

The TACO code reads its models and data from `/blob/...`, a folder on the authors' own computer. Each of these paths must later point to a real file in your Drive folder. This step lists them, so that we know how much has to be changed. Lines that are comments are skipped.

Expected printout: one line per found path with its file name and line number, then the total.

In [ ]:
# HELPER FUNCTION
# Going through every Python file in the folder and collecting the lines that contain the given text.
def find_lines_with_text(folder, search_text):
    found_lines = []
    for python_file in sorted(folder.rglob("*.py")):
        file_text = python_file.read_text(encoding="utf-8", errors="replace")
        for line_number, line_text in enumerate(file_text.splitlines(), start=1):
            if search_text in line_text and not line_text.strip().startswith("#"):
                found_lines.append((python_file.relative_to(folder), line_number, line_text.strip()))
    return found_lines

blob_lines = find_lines_with_text(taco_folder, "/blob/")
for file_name, line_number, line_text in blob_lines:
    print(file_name, "line", line_number, ":", line_text)
print("Total lines with /blob/:", len(blob_lines))

#### **Next steps (not written yet)**

1. Download the model and data files from Hugging Face into the Drive folder (needs the file lists).
2. Change the `/blob/` paths in the working copy, and the wrong script name `debug.py` in `reinforce_mbo.sh`.
3. Install the packages, and test whether they work on the Python version of Colab.
4. Tiny test run (2 rounds, batch size 8) with the timing line.

### **References**

1. Yang Z, Su B, Cao C, Wen JR. Regulatory DNA sequence design with reinforcement learning. In: *International Conference on Learning Representations*. 2025. arXiv:2503.07981. Code (MIT licence): https://github.com/yangzhao1230/TACO